# Tuần 3: K-SVD Dictionary Learning + Full Comparison

Tuần này bạn sẽ:
1. Hiểu OMP (Orthogonal Matching Pursuit)
2. Hiểu K-SVD dictionary learning
3. Denoise sử dụng K-SVD
4. So sánh toàn bộ: Gaussian → NLM → BM3D → K-SVD

Timeline: ~3-4 tiếng để chạy tất cả (K-SVD chậm)

## Cell 0: Setup Path

In [ ]:
import sys
from pathlib import Path

notebook_dir = Path.cwd()
src_path = notebook_dir.parent / 'src' if (notebook_dir.parent / 'src').exists() \
           else notebook_dir / 'src'
sys.path.insert(0, str(src_path))

print(f"✓ Added to path: {src_path}")

## Cell 1: Import

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from skimage import data
import pandas as pd

from noise import add_gaussian_noise
from filters import gaussian_filter
from metrics import psnr, ssim
from nlm import nlm_denoise_fast
from bm3d_wrapper import bm3d_denoise
from sparse_coding import omp, batch_omp
from dictionary_learning import ksvd
from ksvd_denoising import ksvd_denoise, visualize_atoms

img = data.camera().astype(np.uint8)
print(f"✓ Ảnh: {img.shape}, range [{img.min()}, {img.max()}]")

## Cell 2: Hiểu OMP qua Ví Dụ Đơn Giản

In [ ]:
# Tạo dictionary đơn giản: 5D, 3 atoms
np.random.seed(42)
D_toy = np.random.randn(5, 3)
D_toy = D_toy / np.linalg.norm(D_toy, axis=0)  # normalize

# Tạo signal: combination của 2 atoms + noise
y_true = np.zeros(3)
y_true[[0, 2]] = [2.0, -1.5]  # chỉ 2 atom: sparse
y = D_toy @ y_true + 0.1 * np.random.randn(5)

# Run OMP
x_omp = omp(y, D_toy, sparsity=2)

print("\n=== OMP Example ===")
print(f"True sparse code:     {y_true}")
print(f"OMP result:           {x_omp}")
print(f"Reconstruction error: {np.linalg.norm(y - D_toy @ x_omp):.6f}")
print(f"Non-zero coefficients: {np.sum(x_omp != 0)} (target: 2)")

# Visualize
fig, ax = plt.subplots(figsize=(10, 4))
x_pos = np.arange(3)
width = 0.35

ax.bar(x_pos - width/2, y_true, width, label='True', alpha=0.7)
ax.bar(x_pos + width/2, x_omp, width, label='OMP result', alpha=0.7)

ax.set_xlabel('Atom index')
ax.set_ylabel('Coefficient')
ax.set_title('OMP Sparse Representation')
ax.set_xticks(x_pos)
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Cell 3: K-SVD Learning trên Synthetic Data

In [ ]:
# Generate synthetic training data
np.random.seed(42)
n_dim = 64        # patch dimension (8×8)
n_patches = 1000  # training patches
dict_size = 128   # dictionary size

# Create training data
Y_train = np.random.randn(n_dim, n_patches)

print(f"Training data shape: {Y_train.shape}")
print(f"Running K-SVD...")

D, X = ksvd(Y_train, dict_size=dict_size, iterations=5, sparsity=3, verbose=True)

print(f"\nDictionary shape: {D.shape}")
print(f"Sparse codes shape: {X.shape}")
print(f"Average sparsity: {np.mean(np.sum(X != 0, axis=0)):.2f} non-zero coefficients")

# Visualize atoms
fig, axes = plt.subplots(4, 4, figsize=(8, 8))
for idx in range(16):
    ax = axes[idx // 4, idx % 4]
    atom = D[:, idx].reshape(8, 8)
    ax.imshow(atom, cmap='gray')
    ax.set_title(f"Atom {idx}", fontsize=8)
    ax.axis('off')

plt.suptitle(f"Learned Dictionary (dict_size={dict_size})", fontsize=12)
plt.tight_layout()
plt.savefig('../results/week3_learned_atoms.png', dpi=150, bbox_inches='tight')
plt.show()

## Cell 4: K-SVD Denoising (Optional - Chậm!)

In [ ]:
# WARNING: K-SVD denoising chậm (5-10 phút trên CPU)
# Bạn có thể skip cell này nếu muốn, hoặc chạy trên ảnh nhỏ

sigma = 25
noisy = add_gaussian_noise(img, sigma=sigma)

print(f"Noisy PSNR: {psnr(img, noisy):.2f} dB")
print(f"\nRunning K-SVD denoising (may take several minutes)...")

# Run K-SVD
ksvd_result, D_full, X_full = ksvd_denoise(
    noisy,
    dict_size=128,    # nhỏ để nhanh (thường 256)
    patch_size=8,
    iterations=5,     # nhỏ để nhanh (thường 10)
    sparsity=3
)

ksvd_psnr = psnr(img, ksvd_result)
print(f"\n✓ K-SVD PSNR: {ksvd_psnr:.2f} dB")

## Cell 5: So Sánh Tất Cả Phương Pháp

In [ ]:
sigma = 25
noisy = add_gaussian_noise(img, sigma=sigma)

# Run all methods
print("Running all denoising methods...")

results = {}

# Method 1: Gaussian
print("  Gaussian...", end='')
gaussian_result = gaussian_filter(noisy, 5, 1.5)
results['Gaussian'] = {
    'image': gaussian_result,
    'psnr': psnr(img, gaussian_result),
    'ssim': ssim(img, gaussian_result)
}
print(f" PSNR={results['Gaussian']['psnr']:.2f}")

# Method 2: NLM
print("  NLM...", end='')
nlm_result = nlm_denoise_fast(noisy, h=sigma)
results['NLM'] = {
    'image': nlm_result,
    'psnr': psnr(img, nlm_result),
    'ssim': ssim(img, nlm_result)
}
print(f" PSNR={results['NLM']['psnr']:.2f}")

# Method 3: BM3D
print("  BM3D...", end='')
bm3d_result = bm3d_denoise(noisy, sigma_psd=sigma)
results['BM3D'] = {
    'image': bm3d_result,
    'psnr': psnr(img, bm3d_result),
    'ssim': ssim(img, bm3d_result)
}
print(f" PSNR={results['BM3D']['psnr']:.2f}")

# Method 4: K-SVD (if available)
if 'ksvd_result' in locals():
    results['K-SVD'] = {
        'image': ksvd_result,
        'psnr': psnr(img, ksvd_result),
        'ssim': ssim(img, ksvd_result)
    }
    print(f"  K-SVD already done: PSNR={results['K-SVD']['psnr']:.2f}")
else:
    print("  K-SVD (skipped - run Cell 4 if interested)")

print("\n✓ All methods completed")

## Cell 6: Visualization

In [ ]:
# Create comparison figure
n_methods = len(results)
fig, axes = plt.subplots(2, 3, figsize=(15, 10))

# Row 1
axes[0, 0].imshow(img, cmap='gray')
axes[0, 0].set_title('Original')
axes[0, 0].axis('off')

axes[0, 1].imshow(noisy, cmap='gray')
axes[0, 1].set_title(f'Noisy (σ={sigma})\nPSNR={psnr(img, noisy):.2f} dB')
axes[0, 1].axis('off')

axes[0, 2].imshow(results['Gaussian']['image'], cmap='gray')
axes[0, 2].set_title(f"Gaussian\nPSNR={results['Gaussian']['psnr']:.2f} dB")
axes[0, 2].axis('off')

# Row 2
method_names_row2 = ['NLM', 'BM3D', 'K-SVD']
for col, method in enumerate(method_names_row2):
    if method in results:
        axes[1, col].imshow(results[method]['image'], cmap='gray')
        axes[1, col].set_title(f"{method}\nPSNR={results[method]['psnr']:.2f} dB")
    else:
        axes[1, col].text(0.5, 0.5, f"{method}\n(not run)", 
                          ha='center', va='center', transform=axes[1, col].transAxes)
        axes[1, col].set_title(method)
    axes[1, col].axis('off')

plt.suptitle(f"Tuần 3: All Methods Comparison (σ={sigma})", fontsize=14)
plt.tight_layout()
plt.savefig('../results/week3_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

## Cell 7: Benchmark Table

In [ ]:
# Create summary table
summary = []
for method, data in results.items():
    summary.append({
        'Method': method,
        'PSNR (dB)': round(data['psnr'], 2),
        'SSIM': round(data['ssim'], 4)
    })

df = pd.DataFrame(summary)
df = df.sort_values('PSNR (dB)', ascending=False).reset_index(drop=True)

print("\n" + "="*60)
print(f"BENCHMARK (σ={sigma})")
print("="*60)
print(df.to_string(index=False))
print("="*60)

df.to_csv('../results/week3_benchmark.csv', index=False)
print("\n✓ Saved to week3_benchmark.csv")

## Cell 8: Detailed Comparison & Analysis

In [ ]:
# Detailed analysis
print("\n" + "="*60)
print("DETAILED ANALYSIS")
print("="*60)

for method in df['Method'].values:
    data = results[method]
    psnr_val = data['psnr']
    ssim_val = data['ssim']
    improvement = psnr_val - psnr(img, noisy)
    
    print(f"\n{method}:")
    print(f"  PSNR: {psnr_val:.2f} dB (improvement: {improvement:.2f} dB)")
    print(f"  SSIM: {ssim_val:.4f}")

# Insights
print("\n" + "="*60)
print("KEY INSIGHTS")
print("="*60)

best_method = df.iloc[0]['Method']
print(f"\n1. Best method: {best_method}")
print(f"   → Highest PSNR: {df.iloc[0]['PSNR (dB)']} dB")

print(f"\n2. Speed comparison (subjective):")
print(f"   Fastest:  Gaussian filter (instant)")
print(f"   Fast:     NLM (seconds)")
print(f"   Slow:     BM3D (seconds to minutes)")
print(f"   Slowest:  K-SVD (minutes)")

print(f"\n3. Trade-off analysis:")
print(f"   Gaussian: fast but lower quality")
print(f"   NLM:      balance of speed and quality")
print(f"   BM3D:     best traditional method")
print(f"   K-SVD:    competitive with BM3D, learned model")

## Cell 9: Summary & Next Steps

In [ ]:
print("\n" + "="*60)
print("TUẦN 3 SUMMARY")
print("="*60)

print("""
    ✓ Đã học:
      • OMP (Orthogonal Matching Pursuit)
      • K-SVD Dictionary Learning
      • Sparse representation cho denoising
      • So sánh 4 methods: Gaussian, NLM, BM3D, K-SVD
    
    🎯 Mục tiêu tuần 3:
      • Hiểu sparse models vs end-to-end learning
      • Chuẩn bị cho tuần 5 (DnCNN)
    
    📊 Thống kê:
      • Gaussian filter: simple, fast, low quality
      • NLM:            adaptive, good balance
      • BM3D:           SOTA traditional method
      • K-SVD:          learned dictionary, competitive
    
    🚀 Tuần 4:
      • Benchmark chi tiết toàn bộ 4 methods
      • Viết báo cáo technical
      • Chuẩn bị GitHub repo
    
    🎓 Tuần 5:
      • DnCNN: end-to-end learning
      • So sánh traditional vs deep learning
      • Viết "Future Work" hướng research
""")

print("="*60)